# 05 | MEDINA: complemento de R4

**Papel nesta entrega:** Principal: complemento executado de R4

**Origem:** https://drive.google.com/file/d/1jDQh0V6xTbNAsmHYu20YFedGOnjIgeKW/view

Copia de publicacao organizada em 7 out. 2026. Saidas, contadores e metadados de sessao foram removidos; isso nao significa que a campanha original nao tenha sido executada. Nenhum resultado novo foi produzido na curadoria. O registro de proveniencia relaciona codigo, fonte e alteracoes de configuracao.

Os seletores de retomada/abertura automatica foram colocados em estado seguro nesta copia. Ative a etapa pretendida somente apos ler o protocolo; as quatro R4 ja foram consultadas no historico do trabalho.

Usa descoberta_congelada.json e os 49 P95 da campanha de desenvolvimento. Lista de prioridades vazia: repeticao nao aplicavel. Nao treina XGBoost nem busca novas prioridades. Os escores salvos desta etapa alimentam a Tabela 4; imagens de R4 nao foram inspecionadas.

# MEDINA: abrir R4 sem repetir o treinamento

Este complemento resolve a retomada de uma sessao encerrada. Ele utiliza a campanha **MEDINA_HepG2_20260930T235739_244794Z** e le diretamente sua descoberta congelada.

**O que sera feito:** aplicar as caracteristicas e os P95 originais aos sete arquivos de R4, normalizando cada placa pelos seus proprios DMSO, e registrar controles avaliaveis, escalas invalidas e escores.

**O que nao sera feito:** recalcular R1-R3, selecionar caracteristicas, procurar novos candidatos, treinar modelos, recalcular SHAP ou modificar os arquivos da campanha original. A lista de desenvolvimento e vazia; sua taxa de repeticao continua nao aplicavel.

O notebook original exigia igualdade completa da versao do Python para retomar os modelos. Este complemento registra a mudanca de patch (3.13.15 para 3.13.16, quando aplicavel), preserva NumPy/pandas e nao carrega modelos. Nao disfarca essa mudanca editando o protocolo antigo.

**Uso:** execute as cinco secoes em ordem em uma sessao nova. A primeira pode solicitar reinicio apos instalar dependencias. Os caminhos ja estao preenchidos. Nao execute a secao 19 do caderno antigo para fechar este complemento: o relatorio e o ZIP sao produzidos aqui.

**Teste da entrega:** funcoes numericas e metadados testados com dados sinteticos separados; arquivos reais de congelamento conferidos. A campanha real R4 sera executada por voce no Colab.


## 1. Preparar somente as bibliotecas utilizadas

In [ ]:
import sys, subprocess, builtins
from importlib.metadata import version, PackageNotFoundError
if sys.version_info[:2] != (3,13):
    raise RuntimeError('Use uma sessao Colab com Python 3.13. Nao modifique o protocolo antigo.')
if getattr(builtins, '_medina_r4_reiniciar', False):
    raise RuntimeError('Reinicie a sessao pelo menu e execute novamente esta celula.')
esperadas={'numpy':'2.3.5','pandas':'2.2.3'}
def instalada(n):
    try: return version(n)
    except PackageNotFoundError: return None
faltam=[f'{n}=={v}' for n,v in esperadas.items() if instalada(n)!=v]
if faltam:
    builtins._medina_r4_reiniciar=True
    subprocess.check_call([sys.executable,'-m','pip','install','--only-binary=:all:']+faltam)
    raise RuntimeError('REINICIO NECESSARIO: reinicie a sessao e repita esta celula.')
for n,v in esperadas.items():
    m=sys.modules.get(n)
    if m is not None and getattr(m,'__version__',None)!=v:
        builtins._medina_r4_reiniciar=True
        raise RuntimeError('Biblioteca antiga na memoria: reinicie a sessao e repita esta celula.')
import numpy as np
import pandas as pd
print('AMBIENTE PRONTO. Python:',sys.version.split()[0], '| NumPy:',np.__version__,'| pandas:',pd.__version__)


## 2. Conectar ao Drive e indicar a campanha concluida
A pasta original sera somente lida. Resultados novos ficarao em uma pasta irma, identificada como complemento_R4.

In [ ]:
from pathlib import Path
from google.colab import drive
drive.mount('/content/drive')
CAMPANHA = Path('/content/drive/MyDrive/mba_tcc/resultados_MEDINA_HepG2/'
                'didatico_generalizavel/MEDINA_HepG2_20260930T235739_244794Z')
AUTORIZAR_R4 = False
print('Campanha preservada:',CAMPANHA)


## 3. Funcoes da aplicacao

As funcoes de leitura, normalizacao, RMS/dominancia e auditoria foram mantidas do caderno MEDINA. Elas nao fazem selecao de caracteristicas. Para evitar carregar matrizes desnecessarias, sao lidas numericamente somente as 618 medidas dos sete blocos congelados; o calculo de cada medida e independente das demais nessa normalizacao.

Uma escala MAD invalida gera valor nao avaliavel. A funcao nunca substitui um P95 por um valor estimado em R4. O codigo adicional confere arquivos, organiza a aplicacao e produz o relatorio. Este complemento e propositalmente exclusivo da auditoria do MEDINA com lista congelada vazia.


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
from decimal import Decimal, InvalidOperation
import csv, io, json, hashlib, zipfile, warnings, re, platform
from importlib.metadata import version, PackageNotFoundError
import numpy as np
import pandas as pd

def exigir(ok, mensagem):
    if not bool(ok):
        raise ValueError(mensagem)

def sha256(path):
    h = hashlib.sha256()
    with Path(path).open('rb') as f:
        for pedaco in iter(lambda: f.read(1024 * 1024), b''):
            h.update(pedaco)
    return h.hexdigest()

def gravar_json(path, obj):
    def converter(x):
        if isinstance(x, np.ndarray): return x.tolist()
        if isinstance(x, np.generic): return x.item()
        if isinstance(x, Path): return str(x)
        raise TypeError(type(x).__name__)
    Path(path).write_text(json.dumps(obj, ensure_ascii=False, indent=2,
                                    default=converter), encoding='utf-8')

def controles(meta):
    return meta['composto'].str.casefold().isin(CONTROLES)

def dose_chave(valor):
    try:
        numero = Decimal(str(valor).strip())
    except InvalidOperation:
        raise ValueError('Dose nao numerica: ' + str(valor))
    exigir(numero.is_finite() and numero >= 0, 'Dose ausente, negativa ou nao finita.')
    return format(numero.normalize(), 'f')

def poco_padrao(valor):
    m = re.fullmatch(r'([A-Za-z]+)0*([1-9][0-9]*)', str(valor).strip())
    exigir(m is not None, 'Poco invalido: ' + str(valor))
    return m[1].upper() + str(int(m[2])).zfill(2)

def padronizar_metadados_brutos(quadro, path):
    """Separar B1001_R1 em placa=B1001 e replica=R1, sem usar valores medidos."""
    quadro = quadro.copy()
    for col in ['placa', 'replica', 'poco']:
        exigir(col in quadro and quadro[col].notna().all(), 'Chave ausente: ' + str(path))
        quadro[col] = quadro[col].astype(str).str.strip()
    # Nos CSVs desses dois locais, Metadata_Batch NAO e o numero da replica.
    quadro['placa_na_fonte'] = quadro['placa']
    quadro['lote_na_fonte'] = quadro['replica']
    partes = quadro['placa'].str.extract(r'^(B[0-9]+)_(R[0-9]+)$')
    exigir(partes.notna().all().all(), 'Esperado Metadata_Plate como B1001_R1: ' + path.name)
    exigir(quadro['lote_na_fonte'].eq(CFG['lote_bruto_esperado']).all(),
           'Lote/linhagem inesperado no arquivo: ' + path.name)
    quadro['placa'] = partes[0]
    quadro['replica'] = partes[1]
    arquivo = re.search(r'_(B[0-9]+)_(R[0-9]+)_CP_Profiles_Aggregated[.]csv$', path.name)
    exigir(arquivo is not None, 'Nome de arquivo sem placa e replica reconheciveis: ' + path.name)
    exigir(quadro['placa'].eq(arquivo[1]).all() and quadro['replica'].eq(arquivo[2]).all(),
           'Nome do arquivo e metadados internos discordam: ' + path.name)
    return quadro

def ler_documento_csv(path, membro=None):
    if membro:
        with zipfile.ZipFile(path) as z:
            blob = z.read(membro)
        return pd.read_csv(io.BytesIO(blob), dtype=str, keep_default_na=False)
    return pd.read_csv(path, dtype=str, keep_default_na=False)

def ler_valores(paths, nomes):
    metas, matrizes = [], []
    for path in paths:
        bruto = pd.read_csv(path, dtype={c: str for c in colunas_chaves}, low_memory=False)
        bruto = bruto.rename(columns=COLUNAS)
        bruto = padronizar_metadados_brutos(bruto, path)
        for c in CHAVE_POCO: bruto[c] = bruto[c].str.strip()
        bruto['poco'] = bruto['poco'].map(poco_padrao)
        indices = bruto[CHAVE_POCO].merge(mapa, on=CHAVE_POCO, how='left', validate='one_to_one')
        indices['arquivo'] = path.name
        indices['placa_na_fonte'] = bruto['placa_na_fonte'].to_numpy()
        indices['lote_na_fonte'] = bruto['lote_na_fonte'].to_numpy()
        indices['linha_csv'] = np.arange(2, len(bruto) + 2)
        indices['n_objetos'] = (pd.to_numeric(bruto['n_objetos'], errors='raise')
                                 if 'n_objetos' in bruto else np.nan)
        if 'n_objetos' in bruto:
            exigir(np.isfinite(indices.n_objetos).all() and (indices.n_objetos >= 0).all(),
                   'Contagem invalida em ' + path.name)
        matriz = np.full((len(bruto), len(nomes)), np.nan)
        for j, nome in enumerate(nomes):
            if nome in bruto: matriz[:, j] = pd.to_numeric(bruto[nome], errors='raise')
        metas.append(indices)
        matrizes.append(matriz)
    return pd.concat(metas, ignore_index=True), np.concatenate(matrizes)

def normalizar_dmsos(meta, x, nomes):
    z = np.full(x.shape, np.nan)
    registros = []
    for (placa, replica), linhas in meta.groupby(['placa', 'replica']).indices.items():
        linhas = np.asarray(linhas)
        ctrl = linhas[controles(meta.iloc[linhas]).to_numpy()]
        exigir(len(ctrl) >= CFG['min_controles'], f'Controles insuficientes: {placa}/{replica}')
        with warnings.catch_warnings():
            warnings.simplefilter('ignore', RuntimeWarning)
            centro = np.nanmedian(x[ctrl], axis=0)
            escala = CFG['fator_MAD'] * np.nanmedian(np.abs(x[ctrl] - centro), axis=0)
        n = np.isfinite(x[ctrl]).sum(0)
        valido = (np.isfinite(centro) & np.isfinite(escala)
                  & (escala > CFG['epsilon']) & (n >= CFG['min_controles']))
        colunas = np.flatnonzero(valido)
        with np.errstate(over='ignore', divide='ignore', invalid='ignore'):
            z[np.ix_(linhas, colunas)] = ((x[np.ix_(linhas, colunas)] - centro[colunas])
                                          / (escala[colunas] + CFG['epsilon']))
        registros.append(pd.DataFrame({'placa': placa, 'replica': replica, 'medida': nomes,
            'mediana_DMSO': centro, 'MAD_escalado': escala,
            'n_controles_validos': n, 'escala_valida': valido}))
    return z, pd.concat(registros, ignore_index=True)

def rms_e_dominancia(x):
    n = len(x)
    resultado = np.full(n, np.nan)
    fracao = np.full(n, np.nan)
    dominante = np.full(n, -1, dtype=int)
    validos = np.isfinite(x).all(1)
    v = np.abs(x[validos])
    maior = v.max(1)
    proporcao = np.divide(v, maior[:, None], out=np.zeros_like(v), where=maior[:, None] > 0)
    quadrados = proporcao ** 2
    soma = quadrados.sum(1)
    resultado[validos] = maior * np.sqrt(quadrados.mean(1))
    fracao[validos] = np.divide(quadrados.max(1), soma, out=np.zeros(len(v)), where=soma > 0)
    dominante[validos] = np.where(soma > 0, quadrados.argmax(1), -1)
    return resultado, fracao, dominante

def escores(meta, z, referencias=None):
    saida = meta.copy()
    refs, diagnosticos = [], []
    for b, indices in INDICES_BLOCO.items():
        valor, fracao, dom = rms_e_dominancia(z[:, indices])
        saida['RMS_' + b] = valor
        if referencias is None:
            for placa, sub in saida[controles(saida)].groupby('placa'):
                for r in REPS:
                    exigir(np.isfinite(sub.loc[sub.replica == r, 'RMS_' + b]).sum() >= CFG['min_controles'],
                           f'DMSO avaliavel insuficiente: {placa}/{r}/{b}')
                limite = np.quantile(sub['RMS_' + b], CFG['quantil_referencia'], method='linear')
                exigir(np.isfinite(limite) and limite > 0, 'Referencia invalida: ' + placa + '/' + b)
                refs.append({'placa': placa, 'bloco': b, 'P95': limite,
                             'n_DMSO': len(sub), 'n_medidas': len(indices)})
        ref = pd.DataFrame(refs) if referencias is None else referencias
        lookup = ref.loc[ref.bloco == b].set_index('placa').P95
        saida['A_' + b] = valor / saida.placa.map(lookup)
        q = saida[['placa', 'replica', 'poco', 'composto', 'dose', 'n_objetos']].copy()
        q['bloco'] = b
        q['fracao_quadratica_maior_medida'] = fracao
        q['medida_dominante'] = [NOMES[indices[j]] if j >= 0 else '' for j in dom]
        q['alerta_dominancia'] = fracao > CFG['alerta_dominancia_acima']
        q['n_medidas_invalidas'] = (~np.isfinite(z[:, indices])).sum(1)
        diagnosticos.append(q)
    return saida, pd.DataFrame(refs) if referencias is None else referencias.copy(), pd.concat(diagnosticos, ignore_index=True)

def auditar_controles(scores):
    linhas = []
    for (p, r), grupo in scores[controles(scores)].groupby(['placa', 'replica']):
        for b in BLOCOS:
            a = grupo['A_' + b].to_numpy()
            valido = np.isfinite(a)
            linhas.append({'placa': p, 'replica': r, 'bloco': b, 'n_DMSO': len(a),
                'n_avaliaveis': int(valido.sum()), 'n_acima': int((a[valido] > 1).sum()),
                'taxa_pct': 100 * np.mean(a[valido] > 1) if valido.any() else np.nan,
                'mediana_A': np.median(a[valido]) if valido.any() else np.nan,
                'P95_A': np.quantile(a[valido], .95) if valido.any() else np.nan})
    return pd.DataFrame(linhas)


HASHES_FONTE = {
    'protocolo.json': 'fe58caea7e53f5c1362cc63c39509ffd30ace4735f39d3940f63e6411ce80c1d',
    'descoberta_congelada.json': '74a19737deb8da03306756d88efcb210e85902a97c56da996c31de8bae04ee9a',
    'entradas_sha256.json': '61a919f85ba2d9c6553548f99b9ca4a609cd94817ac1b45301a3c70b459f7040'
}


def ambiente_atual():
    """Registrar versoes reais, sem modificar a versao anotada no protocolo antigo."""
    v = {'python': platform.python_version()}
    for nome in ['numpy','pandas','scipy','scikit-learn','xgboost','shap','rdkit']:
        try: v[nome] = version(nome)
        except PackageNotFoundError: v[nome] = None
    return v


def validar_ambiente(original, atual):
    """A aplicacao usa NumPy/pandas. Exige igualdade dessas bibliotecas e Python major.minor.

    Uma mudanca somente no patch do Python e registrada; nao se promete identidade binaria
    de ambientes. Os regressores nao sao carregados nem treinados neste complemento.
    """
    for nome in ['numpy','pandas']:
        exigir(atual[nome] == original[nome], f'Versao divergente de {nome}: execute a secao 1.')
    exigir(atual['python'].split('.')[:2] == original['python'].split('.')[:2],
           'Use a mesma versao principal/secundaria de Python do protocolo (3.13).')
    return {k: {'original': original.get(k), 'atual': atual.get(k)}
            for k in sorted(set(original) | set(atual)) if original.get(k) != atual.get(k)}


def validar_congelamento(proto, congelado, hash_proto):
    """Conferir a lista vazia e as 49 referencias sem consultar valores de R4."""
    exigir(proto['config']['conjunto'] == 'MEDINA_HepG2', 'Este complemento e exclusivo do MEDINA.')
    exigir(proto['config']['reservada'] == 'R4', 'Reserva diferente de R4.')
    exigir(proto['config']['desenvolvimento'] == ['R1','R2','R3'], 'Desenvolvimento divergente.')
    exigir(congelado['protocolo_sha256'] == hash_proto, 'Descoberta/protocolo nao correspondem.')
    exigir(congelado['candidatos'] == [],
           'Ha candidatos no arquivo: este complemento nao substitui sua avaliacao individual.')
    blocos = proto['config']['canais'] + proto['config']['morfologia']
    exigir(set(congelado['medidas_por_bloco']) == set(blocos), 'Blocos divergentes.')
    nomes = sorted({n for v in congelado['medidas_por_bloco'].values() for n in v})
    refs = pd.DataFrame(congelado['referencias'])
    exigir(not refs.duplicated(['placa','bloco']).any(), 'Referencias duplicadas.')
    exigir(np.isfinite(refs.P95).all() and refs.P95.gt(0).all(), 'P95 invalido.')
    placas = [f'B100{i}' for i in range(1,8)]
    exigir(set(map(tuple,refs[['placa','bloco']].to_numpy())) ==
           {(p,b) for p in placas for b in blocos}, 'Referencias incompletas.')
    for b in blocos:
        n = len(congelado['medidas_por_bloco'][b])
        exigir(n > 0 and refs.loc[refs.bloco.eq(b),'n_medidas'].eq(n).all(),
               'Numero de medidas divergente: ' + b)
    return nomes, refs


def preparar(campanha):
    """Restaurar so os recursos necessarios; nenhuma leitura quantitativa de R1-R3/R4."""
    global CFG, COLUNAS, REPS, BLOCOS, ALVO, CONTROLES, CHAVE_POCO
    global colunas_chaves, mapa, NOMES, INDICES_BLOCO
    origem = Path(campanha)
    exigir(origem.is_dir(), 'Pasta da campanha nao encontrada: ' + str(origem))
    for n,h in HASHES_FONTE.items():
        exigir((origem/n).is_file(), 'Arquivo congelado ausente: ' + n)
        exigir(sha256(origem/n) == h, 'Arquivo congelado diferente: ' + n)
    proto = json.loads((origem/'protocolo.json').read_text())
    congelado = json.loads((origem/'descoberta_congelada.json').read_text())
    entradas = json.loads((origem/'entradas_sha256.json').read_text())
    CFG, COLUNAS = proto['config'], proto['colunas']
    REPS = list(CFG['desenvolvimento'])
    BLOCOS = CFG['canais'] + CFG['morfologia']
    ALVO = CFG['bloco_alvo']
    CONTROLES = {s.casefold() for s in CFG['controles']}
    CHAVE_POCO = ['placa','replica','poco']
    colunas_chaves = [k for k,v in COLUNAS.items() if v in CHAVE_POCO]
    NOMES, referencias = validar_congelamento(proto, congelado, HASHES_FONTE['protocolo.json'])
    pos = {n:i for i,n in enumerate(NOMES)}
    INDICES_BLOCO = {b: np.asarray([pos[n] for n in congelado['medidas_por_bloco'][b]]) for b in BLOCOS}
    atual = ambiente_atual()
    diferencas = validar_ambiente(proto['versoes'], atual)
    fonte = Path(proto['fontes_zip'])
    exigir(fonte.is_file() and str(fonte) in entradas, 'Pacote documental nao localizado.')
    exigir(sha256(fonte) == entradas[str(fonte)] == proto['fontes_sha256_declarado'], 'Pacote documental alterado.')
    mapa = ler_documento_csv(fonte, proto['membro_mapa']).rename(columns=COLUNAS)
    necessarias = CHAVE_POCO + ['composto','dose']
    exigir(set(necessarias) <= set(mapa), 'Mapa sem colunas obrigatorias.')
    mapa = mapa[necessarias].copy()
    for c in necessarias:
        mapa[c] = mapa[c].str.strip()
        exigir(mapa[c].ne('').all(), 'Mapa com valor vazio: ' + c)
    mapa['poco'] = mapa.poco.map(poco_padrao)
    mapa['dose_chave'] = mapa.dose.map(dose_chave)
    exigir(not mapa.duplicated(CHAVE_POCO).any(), 'Mapa com pocos duplicados.')
    arquivos = sorted(Path(p) for p in entradas if re.search(r'_B[0-9]+_R4_CP_Profiles_Aggregated[.]csv$', Path(p).name))
    exigir(len(arquivos) == 7, 'Nao foram encontrados os sete arquivos R4 no inventario congelado.')
    for path in arquivos:
        exigir(path.parent == Path(proto['dados']), 'Arquivo de outro local.')
        exigir(path.is_file(), 'Arquivo R4 ausente: ' + str(path))
    # Hashing nao interpreta os dados. A leitura numerica fica na secao 5.
    guardados = {str(origem/n):h for n,h in HASHES_FONTE.items()}
    guardados[str(fonte)] = entradas[str(fonte)]
    for p in arquivos:
        exigir(sha256(p) == entradas[str(p)], 'Arquivo R4 diferente do inventario: ' + p.name)
        guardados[str(p)] = entradas[str(p)]
    print('PREPARACAO CONCLUIDA: referencias originais; 7 arquivos R4; 0 candidatos congelados.')
    print('Python original:',proto['versoes']['python'],'| atual:',atual['python'])
    print('Mudancas registradas:',diferencas or 'nenhuma')
    print('R1-R3 nao recalculadas. Nenhum modelo carregado. R4 ainda nao lida numericamente.')
    return {'origem':origem,'protocolo':proto,'congelado':congelado,
            'referencias':referencias,'arquivos':arquivos,'hashes':guardados,
            'ambiente_atual':atual,'diferencas_ambiente':diferencas}


def executar_r4(estado, autorizar=False):
    """Aplicar referencias congeladas a R4; saida nova e separada, sem alterar a campanha."""
    exigir(autorizar, 'A leitura numerica de R4 exige AUTORIZAR_R4 = True.')
    for p,h in estado['hashes'].items(): exigir(sha256(p) == h, 'Fonte mudou: ' + p)
    origem = estado['origem']
    identificador = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S_%fZ')
    saida = origem.parent / (origem.name + '_complemento_R4_' + identificador)
    saida.mkdir(exist_ok=False)
    (saida/'tabelas').mkdir()
    registro = {'utc_inicio':datetime.now(timezone.utc).isoformat(),
                'origem':str(origem),'protocolo_original_sha256':HASHES_FONTE['protocolo.json'],
                'descoberta_original_sha256':HASHES_FONTE['descoberta_congelada.json'],
                'declaracao_antes_da_campanha':estado['protocolo']['config']['reserva_ja_consultada'],
                'registro_previo_na_origem':(origem/'reserva_aberta.json').exists(),
                'ambiente_original':estado['protocolo']['versoes'],
                'ambiente_atual':estado['ambiente_atual'],
                'diferencas_ambiente':estado['diferencas_ambiente'],
                'escopo':'Aplicacao/auditoria de R4 do MEDINA. Nenhuma descoberta ou predicao nova.',
                'normalizacao':'Mediana/MAD dos proprios DMSO de R4; referencias P95 de R1-R3 congeladas.',
                'medidas':'Somente as 618 medidas congeladas dos sete blocos principais.',
                'repeticao_candidatos':'nao_aplicavel_sem_candidatos',
                'modelo_treinado':False,'status':'iniciada'}
    gravar_json(saida/'registro_aplicacao_R4.json', registro)
    def salvar(nome, df):
        d=df.copy()
        if 'conjunto' not in d: d.insert(0,'conjunto',CFG['conjunto'])
        d.to_csv(saida/'tabelas'/(nome+'.csv'),index=False)
    try:
        for path in estado['arquivos']:
            with path.open(encoding='utf-8-sig',newline='') as f: cols=next(csv.reader(f))
            exigir(len(cols)==len(set(cols)), 'Colunas duplicadas: ' + path.name)
        meta, x = ler_valores(estado['arquivos'], NOMES)
        exigir(meta.replica.eq('R4').all(), 'Leitura incluiu outra replica.')
        exigir(meta[['composto','dose']].notna().all().all(), 'Poco sem identidade no mapa.')
        exigir(not meta.duplicated(CHAVE_POCO).any(), 'Poco repetido.')
        esperado=mapa[mapa.replica.eq('R4')]
        exigir(set(meta.placa)==set(esperado.placa), 'Placa ausente na reserva.')
        ausencia=esperado.merge(meta[CHAVE_POCO],on=CHAVE_POCO,how='left',indicator=True)
        salvar('15_pocos_ausentes_reserva',ausencia[ausencia['_merge'].eq('left_only')].drop(columns='_merge'))
        exigir(not np.isinf(x).any(), 'Valores brutos infinitos: revise os dados antes de prosseguir.')
        if CFG['min_objetos_inclusao'] is not None:
            exigir(meta.n_objetos.notna().all(), 'Contagem ausente.')
            incluir=meta.n_objetos.ge(CFG['min_objetos_inclusao']).to_numpy()
            salvar('15_exclusoes_reserva_contagem',meta.loc[~incluir])
            meta,x=meta.loc[incluir].reset_index(drop=True),x[incluir]
        exigir(len(meta)>0, 'Nenhum perfil restante.')
        meta['alerta_poucos_objetos']=meta.n_objetos.lt(CFG['alerta_objetos_abaixo'])
        z,parametros=normalizar_dmsos(meta,x,NOMES)
        scores,_,dominancia=escores(meta,z,estado['referencias'])
        auditoria=auditar_controles(scores)
        mapa_blocos={n:b for b,v in estado['congelado']['medidas_por_bloco'].items() for n in v}
        parametros['bloco']=parametros.medida.map(mapa_blocos)
        parametros['selecionada']=True
        salvar('15_parametros_reserva',parametros)
        salvar('15_medidas_selecionadas_invalidas',parametros[~parametros.escala_valida])
        salvar('15_escores_reserva',scores)
        salvar('15_dominancia_reserva',dominancia)
        salvar('15_DMSO_reserva',auditoria)
        salvar('15_referencias_desenvolvimento_utilizadas',estado['referencias'])
        # Tabela vazia com cabecalho: nao converte ausencia de candidatos em negativos.
        colunas=['id_condicao','composto','placa','dose','estado_reserva',
                 'A_alvo_desenvolvimento','A_alvo_reserva','cosseno_min_reserva','motivos']
        salvar('15_avaliacao_candidatos_reserva',pd.DataFrame(columns=colunas))
        resumo=auditoria.groupby('bloco',sort=False)[['n_DMSO','n_avaliaveis','n_acima']].sum().reset_index()
        resumo['taxa_pct']=np.where(resumo.n_avaliaveis.gt(0),100*resumo.n_acima/resumo.n_avaliaveis,np.nan)
        salvar('15_resumo_DMSO_reserva',resumo)
        for p,h in estado['hashes'].items(): exigir(sha256(p)==h, 'Fonte modificada durante a execucao: '+p)
        registro.update({'status':'concluida','utc_fim':datetime.now(timezone.utc).isoformat(),
                         'n_perfis_R4':len(meta),'n_DMSO':int(controles(meta).sum()),
                         'n_candidatos_congelados':0,'n_blocos_placas_nao_avaliaveis':int(auditoria.n_avaliaveis.eq(0).sum()),
                         'taxa_confirmacao':None,'fontes_utilizadas_inalteradas':True})
        gravar_json(saida/'registro_aplicacao_R4.json',registro)
        # O relatorio anterior e copiado para contexto; nao e reescrito no local original.
        if (origem/'relatorio.md').is_file():
            (saida/'relatorio_desenvolvimento_original.md').write_bytes((origem/'relatorio.md').read_bytes())
        linhas=['# MEDINA HepG2: complemento de auditoria de R4',
                f'Campanha de desenvolvimento: {origem.name}.',
                f'Perfis R4 incluidos: {len(meta)}. Controles DMSO: {int(controles(meta).sum())}.',
                'As 618 medidas dos sete blocos e os 49 P95 foram lidos da descoberta congelada.',
                'A normalizacao local utiliza DMSO de R4; nao refaz selecao nem limites P95.',
                '## Controles de R4 por bloco',resumo.to_string(index=False),
                '## Interpretacao',
                '0 candidatos no desenvolvimento. Repeticao de candidatos: nao aplicavel, nao 0%.',
                'Taxas usam somente controles avaliaveis. Ausencia de escore nao e resultado negativo.',
                'Excedencias de tratamentos sao saidas numericas de aplicacao; nao foram usadas para buscar novos candidatos.',
                'Nenhum modelo carregado ou treinado; nenhum SHAP calculado; nenhuma imagem inspecionada.',
                '## Ambiente e rastreabilidade',
                'Python original: '+estado['protocolo']['versoes']['python']+'; atual: '+estado['ambiente_atual']['python']+'.',
                'NumPy e pandas mantidos nas versoes de desenvolvimento. Diferencas adicionais registradas, sem alterar o protocolo original.',
                'O codigo de normalizacao, RMS/dominancia e auditoria de controles foi mantido do caderno de origem.',
                'A igualdade entre as duas versoes de Python nao foi pressuposta como identidade de ambientes.',
                'Resultados originais de desenvolvimento e modelos nao foram sobrescritos.']
        relatorio='\n\n'.join(linhas)
        (saida/'relatorio.md').write_text(relatorio,encoding='utf-8')
        gravar_json(saida/'fontes_sha256.json',estado['hashes'])
        gravar_json(saida/'fontes_congeladas.json',{'protocolo_original':estado['protocolo'],'descoberta_original':estado['congelado']})
        if globals().get('CODIGO_RECUPERACAO'):
            (saida/'codigo_executado.py').write_text(CODIGO_RECUPERACAO,encoding='utf-8')
        gravar_json(saida/'arquivos_sha256.json',{str(p.relative_to(saida)):sha256(p) for p in saida.rglob('*') if p.is_file()})
        pacote=saida.with_name(saida.name+'_leitura.zip')
        with zipfile.ZipFile(pacote,'w',compression=zipfile.ZIP_DEFLATED) as zz:
            for p in sorted(saida.rglob('*')):
                if p.is_file(): zz.write(p,p.relative_to(saida))
        print(relatorio)
        print('\nRESULTADOS:',saida,'\nPACOTE DE LEITURA:',pacote)
        return {'saida':saida,'pacote':pacote,'auditoria':auditoria,'resumo':resumo,'relatorio':relatorio}
    except Exception as erro:
        registro.update({'status':'falhou','erro':str(erro)})
        gravar_json(saida/'registro_aplicacao_R4.json',registro)
        raise


## 4. Conferir as fontes e restaurar as referencias
Esta etapa restaura as variaveis de que a aplicacao necessita. Ela confere hashes e o mapa; nao interpreta valores quantitativos de R4.

In [ ]:
# Arquivar a celula de funcoes quando o historico do Colab estiver disponivel.
# A aplicacao nao depende dessa copia: as funcoes ja foram definidas na secao 3.
CODIGO_RECUPERACAO = None
try:
    historico = get_ipython().history_manager.input_hist_raw
    CODIGO_RECUPERACAO = next(
        (s for s in reversed(historico[:-1]) if s.startswith('from pathlib import Path')
         and 'def executar_r4(' in s), None)
except (NameError, AttributeError):
    pass
ESTADO = preparar(CAMPANHA)


## 5. Aplicar a R4 e exportar o complemento

A execucao desta celula com `AUTORIZAR_R4=True` libera a leitura dos perfis R4. Os tratamentos recebem escores, mas nao passam por uma nova busca de candidatos. Consulte principalmente `15_DMSO_reserva.csv`, `15_medidas_selecionadas_invalidas.csv` e o relatorio.

A taxa de excedencia usa somente os controles avaliaveis como denominador; ausencia de controles avaliaveis aparece como valor ausente, nao 0%. Um P95 local diferente nao e, isoladamente, evidencia de artefato ou de biologia.


In [ ]:
RESULTADO = executar_r4(ESTADO, autorizar=AUTORIZAR_R4)
display(RESULTADO['auditoria'].query("bloco == 'Mito'"))
